In [12]:
import os
import torch
from torch import nn
from skimage.transform import resize
from skimage.io import imread
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

import torch.nn as nn

Шаг 1. Загрузка и подготовка данных

1. Для начала мы скачаем датасет: [ADDI project](https://www.fc.up.pt/addi/ph2%20database.html).

<table>
    <tr>
        <td>
            <img src="PH2Dataset/PH2 Dataset images/IMD063/IMD063_Dermoscopic_Image/IMD063.bmp">
        </td>
        <td>
            <img src="PH2Dataset/PH2 Dataset images/IMD063/IMD063_lesion/IMD063_lesion.bmp">
        </td>
    </tr>
</table>

2. Разархивируем .rar файл.

Это фотографии двух типов **поражений кожи:** меланома и родинки.
В каждой папке IM D_00% находится информация об одном наблюдении. В подпапках соответствующего наблюдения - изображение в исходном виде и сегментированное изображение.

In [ ]:
!gdown 1T_RPkPP0jeWwK8L1UrmBw8V30eD7v6Ql

In [ ]:
get_ipython().system_raw("unrar x PH2Dataset.rar")

Изображения имеют разные размеры. Для однообразия и удобства работы создам кастомный класс для датасета с учетом размера изображений $256\times256 $ пикселей


In [11]:
class CustomDataset(TensorDataset):
    '''
        Some DOCstring need to be added
    '''
    
    def __init__(self,
                 root :str = 'PH2Dataset',
                 folder_name :str = 'PH2 Dataset images',
                 pic_size :tuple = (256, 256)):

        self.pic_size = pic_size
        self.images_paths = []
        self.lesions_paths = []

        # проходим по директориям и собираем пути к файлам
        for root, dirs, files in os.walk(os.path.join(root, folder_name)):
            if root.endswith(_Dermoscopic_Image):
                self.images_paths.append(os.path.join(root, files[0]))
            
                if root.endswith(_lesion):
                    self.lesions_paths.append(os.path.join(root, files[0]))
                
    def __len__(self):
        '''
        возращает количество изображений
        '''
        
        return len(self.images_paths)
        
    def __getitem__(self, idx):
        '''
        получаем очередное изображение -> трансформируем
        '''

        # transform.resize() для исходного изображения
        img_feature = imread(self.images_paths[idx])        
        img_feature = resize(img_feature,
                             self.pic_size,
                             mode=constant,
                             anti_aliasing=True,)

        # transform.resize() для сегментированного (обучающего) изображения
        img_target = imread(self.lesions_paths[idx])
        img_target = resize(img_target,
                            self.pic_size,
                            mode=constant,
                            anti_aliasing=False,) > 0.5
        
        return torch.from_numpy(
            np.array(
                np.rollaxis(
                    img_feature, 2, 0
                ), np.float32)
        ), torch.from_numpy(np.array(img_target, np.float32))

Популярным лоссом для бинарной сегментации является *бинарная кросс-энтропия*, которая задается следующим образом:

$$\mathcal L_{BCE}(y, \hat y) = -\sum_i \left[y_i\log\sigma(\hat y_i) + (1-y_i)\log(1-\sigma(\hat y_i))\right] \space [1]$$

где $y$ это  таргет желаемого результата и $\hat y$ является выходом модели. $\sigma$ - это [*логистическая* функция](https://en.wikipedia.org/wiki/Sigmoid_function), который преобразует действительное число $\mathbb R$ в вероятность $[0,1]$.

Однако эта потеря страдает от проблем численной нестабильности. Самое главное, что $\lim_{x\rightarrow0}\log(x)=\infty$ приводит к неустойчивости в процессе оптимизации. Рекомендуется посмотреть следующее [упрощение](https://www.tensorflow.org/api_docs/python/tf/nn/sigmoid_cross_entropy_with_logits). Эта функция эквивалентна первой и не так подвержена численной неустойчивости:

$$\mathcal L_{BCE} = \hat y - y\hat y + \log\left(1+\exp(-\hat y)\right) \space [2]$$

In [13]:
def bce_loss(y_pred, y_real):
  # TODO    
  return -torch.sum(y_real * nn.functional.logsigmoid(y_pred) + (1 - y_real) * torch.log(1 - torch.sigmoid(y_pred)))

def bce_true(y_pred, y_real):
  # TODO
  return torch.sum(y_pred - y_pred*y_real + torch.log(1 + torch.exp(-y_pred)))

In [ ]:
y_pred = torch.randn(3, 2, requires_grad=False)
y_true = torch.rand(3, 2, requires_grad=False)

print(f'BCE loss from scratch bce_loss             = {bce_loss(y_pred, y_true)}')
print(f'BCE loss честно посчитанный                = {bce_true(y_pred, y_true)}')
print(f'BCE loss from torch bce_torch              = {bce_torch(torch.sigmoid(y_pred), y_true)}')
print(f'BCE loss from torch with logits bce_torch  = {bce_torch_with_logits(y_pred, y_true)}')


In [ ]:
assert np.isclose(bce_loss(y_pred, y_true), bce_torch(torch.sigmoid(y_pred), y_true))
assert np.isclose(bce_loss(y_pred, y_true), bce_torch_with_logits(y_pred, y_true))
assert np.isclose(bce_true(y_pred, y_true), bce_torch(torch.sigmoid(y_pred), y_true))
assert np.isclose(bce_true(y_pred, y_true), bce_torch_with_logits(y_pred, y_true))

In [ ]:
# Параметрами блока будут:
# - количество каналов на входе
# - количество каналов на выходе
# - глубина блока (2 или 3, по количеству конволюционных слоев)
# - kernel_size и padding
#

class ConvReLU(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1) -> None:
        super(ConvReLU, self).__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c, depth=2, kernel_size=3, padding=1) -> None:
        super(EncoderBlock, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(depth):
            self.layers.append(ConvReLU(in_c if i == 0 else out_c, out_c, kernel_size, padding))
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x, ind = self.pool(x)
        return x, ind

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c, depth=2, kernel_size=3, padding=1, classification=False) -> None:
        super(DecoderBlock, self).__init__()
        self.unpool = nn.MaxUnpool2d(kernel_size=2, stride=2)
        self.layers = nn.ModuleList()
        for i in range(depth):
            if i == depth - 1 and classification:
                self.layers.append(nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding))
            elif i == depth - 1:
                self.layers.append(ConvReLU(in_c, out_c, kernel_size=kernel_size, padding=padding))
            else:
                self.layers.append(ConvReLU(in_c, in_c, kernel_size=kernel_size, padding=padding))

    def forward(self, x, ind):
        x = self.unpool(x, ind)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
class SegNet(nn.Module): 
    def __init__(self, in_channels=3, out_channels=1, features=64) -> None:
        super(SegNet, self).__init__()

        # Encoder
        self.enc0 = EncoderBlock(in_channels, features)
        self.enc1 = EncoderBlock(features, features * 2)
        self.enc2 = EncoderBlock(features * 2, features * 4, depth=3)
        self.enc3 = EncoderBlock(features * 4, features * 8, depth=3)

        # Bottleneck
        self.bottleneck_enc = EncoderBlock(features * 8, features * 8, depth=3) 
        self.bottleneck_dec = DecoderBlock(features * 8, features * 8, depth=3) 

        # Decoder
        self.dec0 = DecoderBlock(features * 8, features * 4, depth=3)
        self.dec1 = DecoderBlock(features * 4, features * 2, depth=3)
        self.dec2 = DecoderBlock(features * 2, features)
        self.dec3 = DecoderBlock(features, out_channels, classification=True) # No activation

    def forward(self, x):
        # encoder
        e0, ind0 = self.enc0(x) 
        e1, ind1 = self.enc1(e0) 
        e2, ind2 = self.enc2(e1) 
        e3, ind3 = self.enc3(e2)

        # bottleneck
        b0, indb = self.bottleneck_enc(e3)
        b1 = self.bottleneck_dec(b0, indb)

        # decoder
        d0 = self.dec0(b1, ind3)
        d1 = self.dec1(d0, ind2)
        d2 = self.dec2(d1, ind1)

        # classification layer
        output = self.dec3(d2, ind0)  
        return output

In [ ]:
def check_the_microfone(model, metric, data):
    '''
    функция проверки выбранной метрики для выбранной модели на выбранном датасете
    '''
    model.eval()  # testing mode
    scores = 0
    for X_batch, Y_label in data:
        Y_pred = torch.sigmoid(model(X_batch.to(device)))
        Y_pred = torch.where(Y_pred > 0.5, 1, 0)
        scores += metric(Y_pred.round(), Y_label.to(device)).mean().item()

    return scores / len(data)

In [ ]:
def train(model, opt, loss_fn, epochs, data_tr, data_val):
    history = {
        "train_loss": [],
        "train_score": [],
        "val_loss": [],
        "val_score": []
    }
    
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=4, gamma=0.5)
    
    X_val, Y_val = data_val
    
    for epoch in range(epochs):
        tic = time()
        
        train_loss = 0
        model.train() 
        for X_batch, Y_batch in data_tr:
            
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            opt.zero_grad()

            # forward
            Y_pred = model(X_batch)
            loss = loss_fn(Y_batch, Y_pred) # forward-pass
            loss.backward()  # backward-pass
            opt.step() 

            
            train_loss += loss / len(data_tr)

        train_score = check_the_microfone(model, iou_score, data_tr)
        history["train_loss"].append(train_loss.item())
        history["train_score"].append(train_score)
      
        scheduler.step()
        toc = time()
        
        val_loss = 0
        
        model.eval()  # testing mode
        with torch.no_grad():
            for X_val, Y_val in data_val:
                X_val = X_val.to(device)
                Y_val = Y_val.to(device)

                Y_pred = model(X_val)
                loss = loss_fn(Y_pred, Y_val)

                # loss
                val_loss += loss / len(data_val)

        val_score = check_the_microfone(model, iou_score, data_val)
        history["val_loss"].append(val_loss.item())
        history["val_score"].append(val_score)

        Y_hat = model(X_val.to(device)).detach().cpu() 
        
        clear_output(wait=True)
        for k in range(6):
            plt.subplot(2, 6, k+1)
            plt.imshow(np.rollaxis(X_val[k].to("cpu").numpy(), 0, 3), cmap='gray')
            plt.title('Real')
            plt.axis('off')

            plt.subplot(2, 6, k+7)
            plt.imshow(Y_hat[k, 0], cmap='gray')
            plt.title('Output')
            plt.axis('off')
        plt.show()
        
        print(f'{epoch+1} / {epochs}: {train_loss}')

    return history


In [ ]:
def predict(model, data):
    model.eval()  
    return np.array([X_batch for X_batch, _ in data])

In [ ]:
result_scores = pd.DataFrame(columns=["Модель", "Loss", "score_train", "score_val", "score_test"])

In [ ]:
segnet_model_bce = SegNet().to(device)

In [ ]:
max_epochs = 50
optim = torch.optim.Adam(segnet_model_bce.parameters(), lr=1e-4)
segnet_history_bce = train(segnet_model_bce, optim, bce_loss, max_epochs, train_dataloader, valid_dataloader)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
   
axes[0].plot(segnet_history_bce["train_loss"], label="train loss")
axes[0].plot(segnet_history_bce["val_loss"], label="val loss")
axes[0].set_xticks(np.arange(0, max_epochs + 1, 5))
axes[0].set_title(f"график изменения bce")
axes[0].legend()
    
axes[1].plot(segnet_history_bce["train_score"], label="train score")
axes[1].plot(segnet_history_bce["val_score"], label="val score")
axes[1].set_xticks(np.arange(0, max_epochs + 1, 5))
axes[1].set_title(f"график изменения iou")
axes[1].legend()

plt.show()